# Latent Factor Model — SVD
Ce notebook implémente et évalue un modèle de facteurs latents (SVD) pour le système de recommandation Cinematch.
Il sert également à générer l'artefact `backend/artifacts/svd_model.pkl` utilisé par le serveur FastAPI.

# Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
import os
import numpy as np
import pandas as pd
import random as rd

from surprise import SVD, accuracy
from surprise.model_selection import cross_validate, train_test_split, LeaveOneOut

from loaders import load_ratings, load_items
from constants import Constant as C
from models import ModelBaseline4, get_top_n

# 1. Chargement des données

In [ ]:
sp_ratings = load_ratings(surprise_format=True)
df_ratings = load_ratings(surprise_format=False)
df_items   = load_items()
trainset   = sp_ratings.build_full_trainset()

print(f"Nombre de ratings     : {trainset.n_ratings}")
print(f"Nombre d'utilisateurs : {trainset.n_users}")
print(f"Nombre de films       : {trainset.n_items}")
print(f"Moyenne globale       : {trainset.global_mean:.4f}")

# 2. Comprendre SVD — Facteurs Latents

L'idée centrale de SVD (Simon Funk SVD) est de décomposer la matrice de ratings **R** en deux matrices de facteurs latents :

$$\hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i$$

- $\mu$ : moyenne globale des ratings  
- $b_u$ : biais utilisateur (cet utilisateur note-t-il plus haut ou plus bas que la moyenne ?)  
- $b_i$ : biais item (ce film est-il mieux ou moins bien noté que la moyenne ?)  
- $p_u \in \mathbb{R}^k$ : vecteur de facteurs latents de l'utilisateur  
- $q_i \in \mathbb{R}^k$ : vecteur de facteurs latents du film  

Le modèle apprend $p_u$ et $q_i$ en minimisant l'erreur quadratique (avec régularisation L2) sur le trainset via SGD.

**Pour un nouvel utilisateur** (non présent dans le trainset), on utilise le **folding-in** :
on fixe les $q_i$ appris et on résout analytiquement pour $p_u$ à partir des ratings qu'il a fournis.

# 3. Explorer le SVD de Surprise (ModelBaseline4)

In [ ]:
# ModelBaseline4 = SVD(n_factors=100, random_state=1)
algo = ModelBaseline4()
algo.fit(trainset)

print(f"n_factors : {algo.n_factors}")
print(f"Forme qi (facteurs items)  : {algo.qi.shape}")
print(f"Forme pu (facteurs users)  : {algo.pu.shape}")
print(f"Forme bi (biais items)     : {algo.bi.shape}")
print(f"Forme bu (biais users)     : {algo.bu.shape}")

In [ ]:
# Faire une prédiction sur le premier élément de l'anti-testset
anti_testset = trainset.build_anti_testset()
first = anti_testset[0]
pred = algo.predict(first[0], first[1])
print(f"Prédiction — user: {pred.uid} | item: {pred.iid} | est: {pred.est:.4f}")

# 4. Optimisation des hyperparamètres — GridSearchCV

Les principaux hyperparamètres de SVD :
- `n_factors` : nombre de dimensions latentes (complexité du modèle)
- `n_epochs` : nombre d'itérations SGD
- `lr_all` : taux d'apprentissage
- `reg_all` : terme de régularisation L2

On utilise `GridSearchCV` de Surprise pour tester toutes les combinaisons par cross-validation (3 folds) et retenir automatiquement la meilleure configuration.

In [ ]:
from surprise.model_selection import GridSearchCV

param_grid = {
    "n_factors": [50, 100, 150],
    "n_epochs":  [20, 30],
    "lr_all":    [0.002, 0.005, 0.01],
    "reg_all":   [0.02, 0.05, 0.1],
}

print(f"Nombre de combinaisons : {3*2*3*3} × 3 folds = {3*2*3*3*3} entraînements")
print("Lancement du GridSearchCV (peut prendre 15–30 min)...\n")

gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=3, n_jobs=-1)
gs.fit(sp_ratings)

best_params = gs.best_params["rmse"]
best_rmse   = gs.best_score["rmse"]
best_mae    = gs.best_score["mae"]

print(f"Meilleurs paramètres : {best_params}")
print(f"Meilleur RMSE        : {best_rmse:.4f}")
print(f"Meilleur MAE         : {best_mae:.4f}")

# 5. Évaluation complète — MAE, RMSE, Hit Rate, Novelty, Diversity

| Métrique | Type | Formule | Sens |
|---|---|---|---|
| **RMSE / MAE** | Split | Erreur prédiction | ↓ mieux |
| **Hit Rate** | LOO | Proportion de hits dans top-N | ↑ mieux |
| **Novelty (rang)** | Full | Rang popularité moyen (*evaluator.ipynb*) | ↑ mieux |
| **Novelty (MIUF)** | Full | $\frac{1}{|R|}\sum_{i \in R} -\log_2\frac{|U_i|}{|U|}$ | ↑ mieux |
| **Diversity (ILD)** | Full | $\frac{1}{|R|(|R|-1)}\sum_{i}\sum_{j} d(i,j)$ | ↑ mieux |

## 5.1 Fonctions de validation (identiques à evaluator.ipynb)

In [ ]:
TOP_N = 40
TEST_SIZE = 0.25

def generate_split_predictions(algo, ratings_dataset):
    tr, te = train_test_split(ratings_dataset, test_size=TEST_SIZE, random_state=42)
    algo.fit(tr)
    return algo.test(te)

def generate_loo_top_n(algo, ratings_dataset):
    loo = LeaveOneOut(n_splits=1, random_state=1)
    for tr, te in loo.split(ratings_dataset):
        algo.fit(tr)
        preds = algo.test(tr.build_anti_testset())
        top_n = get_top_n(preds, n=TOP_N)
    return top_n, te

def generate_full_top_n(algo, ratings_dataset):
    full_tr = ratings_dataset.build_full_trainset()
    algo.fit(full_tr)
    preds = algo.test(full_tr.build_anti_testset())
    return get_top_n(preds, n=TOP_N)

print("Fonctions de validation prêtes.")

## 5.2 Métriques (get_hit_rate et get_novelty copiées de evaluator.ipynb)

In [ ]:
# ── Hit Rate — identique à evaluator.ipynb ────────────────────────────────────
def get_hit_rate(anti_testset_top_n, testset):
    """A hit (1) happens when the movie in the testset has been picked by the top-n recommender."""
    hits = 0
    total_users = len(testset)
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recommendations = [item_id for (item_id, _) in anti_testset_top_n[user_id]]
            if movie_id in recommendations:
                hits += 1
    return hits / total_users if total_users > 0 else 0.0


# ── Novelty (rang) — identique à evaluator.ipynb ──────────────────────────────
def get_novelty_rank(anti_testset_top_n, item_to_rank):
    """Average popularity rank of recommended items (higher = more novel)."""
    total_novelty = 0
    total_users = len(anti_testset_top_n)
    for user_id, recommendations in anti_testset_top_n.items():
        user_sum = sum(
            item_to_rank.get(movie_id, len(item_to_rank))
            for movie_id, _ in recommendations
        )
        total_novelty += user_sum
    return total_novelty / total_users if total_users > 0 else 0.0


# ── Novelty MIUF — formule du cours ───────────────────────────────────────────
# MIUF = (1/|R|) * sum_{i in R} [ -log2(|U_i| / |U|) ]
# |U|   = nombre total d'utilisateurs
# |U_i| = nombre d'utilisateurs ayant noté le film i
def get_novelty_miuf(anti_testset_top_n, df_ratings):
    n_users   = df_ratings[C.USER_ID_COL].nunique()                      # |U|
    item_freq = df_ratings.groupby(C.ITEM_ID_COL)[C.USER_ID_COL].nunique()  # |U_i|

    user_novelties = []
    for uid, recs in anti_testset_top_n.items():
        if not recs:
            continue
        miuf_list = [
            -np.log2(item_freq.get(iid, 1) / n_users)   # -log2(|U_i|/|U|)
            for iid, _ in recs
        ]
        user_novelties.append(np.mean(miuf_list))        # moyenne sur |R| items
    return float(np.mean(user_novelties)) if user_novelties else 0.0


# ── Diversity ILD — formule du cours ──────────────────────────────────────────
# ILD = (1 / |R|(|R|-1)) * sum_{i in R} sum_{j in R} d(i, j)
# d(i,j) = dissimilarité cosinus sur les vecteurs genres binaires normalisés
def build_genre_vectors(df_items):
    all_genres = sorted(set(
        g for genres in df_items[C.GENRES_COL].fillna('').str.split('|')
        for g in genres if g and g != '(no genres listed)'
    ))
    genre_index = {g: i for i, g in enumerate(all_genres)}
    vectors = {}
    for mid, row in df_items.iterrows():
        vec = np.zeros(len(all_genres))
        for g in str(row[C.GENRES_COL]).split('|'):
            if g in genre_index:
                vec[genre_index[g]] = 1.0
        norm = np.linalg.norm(vec)
        vectors[mid] = vec / norm if norm > 0 else vec
    return vectors

def get_diversity_ild(anti_testset_top_n, genre_vectors):
    user_ilds = []
    for uid, recs in anti_testset_top_n.items():
        ids = [iid for iid, _ in recs if iid in genre_vectors]
        R = len(ids)
        if R < 2:
            continue
        vecs = np.array([genre_vectors[i] for i in ids])  # (R x n_genres)
        sim  = vecs @ vecs.T                               # cosine similarities
        # somme de toutes les paires (i,j) avec i != j
        total_dist = np.sum(1 - sim) - R * (1 - 1.0)      # soustrait diagonale (=0)
        user_ilds.append(total_dist / (R * (R - 1)))
    return float(np.mean(user_ilds)) if user_ilds else 0.0


# ── Pré-calcul popularité (pour novelty rang) ─────────────────────────────────
def precompute_item_to_rank(df_ratings):
    movie_counts = df_ratings[C.ITEM_ID_COL].value_counts()
    return movie_counts.rank(ascending=False, method='first').to_dict()

print("Métriques prêtes.")

## 5.3 Calcul de toutes les métriques

In [ ]:
rd.seed(1)
np.random.seed(1)

# Pré-calculs
item_to_rank  = precompute_item_to_rank(df_ratings)
genre_vectors = build_genre_vectors(df_items)

results = {}

# 1. RMSE / MAE
print("[1/4] RMSE / MAE...")
split_preds      = generate_split_predictions(ModelBaseline4(), sp_ratings)
results['RMSE']  = accuracy.rmse(split_preds, verbose=False)
results['MAE']   = accuracy.mae(split_preds,  verbose=False)

# 2. Hit Rate (LOO)
print("[2/4] Hit Rate (LOO)...")
top_n_loo, testset_loo  = generate_loo_top_n(ModelBaseline4(), sp_ratings)
results['Hit Rate']     = get_hit_rate(top_n_loo, testset_loo)

# 3. Novelty (rang) + MIUF + ILD (full trainset)
print("[3/4] Novelty (rang + MIUF) et Diversity (ILD)...")
top_n_full                    = generate_full_top_n(ModelBaseline4(), sp_ratings)
results['Novelty (rang)']     = get_novelty_rank(top_n_full, item_to_rank)
results['Novelty (MIUF)']     = get_novelty_miuf(top_n_full, df_ratings)
results['Diversity (ILD)']    = get_diversity_ild(top_n_full, genre_vectors)

print("[4/4] Terminé.\n")

df_results = pd.DataFrame.from_dict(results, orient='index', columns=['ModelBaseline4 (SVD)'])
display(df_results.round(4))

## 5.4 Interprétation des métriques

### RMSE / MAE — précision de prédiction (↓ mieux)
Mesurent l'écart entre le rating prédit et le rating réel. SVD optimise directement RMSE via SGD — c'est pourquoi il obtient les meilleurs scores parmi les baselines collaboratifs (RMSE ≈ 0.817 dans evaluator.ipynb).

### Hit Rate (↑ mieux)
Proportion d'utilisateurs pour qui le film "caché" (LOO) apparaît dans le top-40. Mesure la capacité à retrouver ce qu'un utilisateur aimerait réellement.

### Novelty (rang) vs Novelty (MIUF) — pourquoi MIUF est plus rigoureux

| Critère | Rang | MIUF |
|---|---|---|
| Sensible à N (taille top-N) | ✗ Oui | ✓ Non |
| Échelle logarithmique (cohérente) | ✗ Non | ✓ Oui |
| Comparable entre catalogues | ✗ Non | ✓ Oui |

La formule MIUF : $\frac{1}{|R|}\sum_{i \in R} -\log_2\frac{|U_i|}{|U|}$ — un film vu par tout le monde → MIUF ≈ 0, un film très rare → MIUF élevé.

### Diversity ILD (↑ mieux)
$$ILD = \frac{1}{|R|(|R|-1)}\sum_{i \in R}\sum_{j \in R} d(i,j) \quad \text{avec} \quad d(i,j) = 1 - \cos(g_i, g_j)$$
$g_i$ = vecteur genre binaire normalisé du film $i$. Valeur entre 0 (tous pareils) et 1 (tous différents).

# 6. Visualiser les facteurs latents (PCA)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

algo_full = ModelBaseline4()
algo_full.fit(trainset)

pca = PCA(n_components=2, random_state=1)
qi_2d = pca.fit_transform(algo_full.qi)
print(f"Variance expliquée par les 2 composantes : {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 6))
plt.scatter(qi_2d[:, 0], qi_2d[:, 1], s=2, alpha=0.4)
plt.title("Facteurs latents items projetés en 2D (PCA)")
plt.xlabel("Composante 1")
plt.ylabel("Composante 2")
plt.tight_layout()
plt.show()

# 7. Démonstration du Folding-In

Pour un **nouvel utilisateur** non présent dans le trainset, on estime son vecteur $p_u$ en résolvant :

$$A \, p_u = b \quad \text{avec} \quad A = Q_{rated}^T Q_{rated} + \lambda I, \quad b = Q_{rated}^T \cdot \text{residuals}$$

où `residuals = ratings - mu - b_rated`.

In [ ]:
LAMBDA = 0.1
user_ratings_demo = {1: 4.0, 2: 3.5, 3: 5.0, 4: 3.0, 5: 4.5}

ts = algo_full.trainset
mu = ts.global_mean

rated = []
for raw_iid, r in user_ratings_demo.items():
    try:
        inner = ts.to_inner_iid(raw_iid)
        rated.append((inner, r))
    except ValueError:
        print(f"  film {raw_iid} inconnu du trainset, ignoré")

inner_ids = [x[0] for x in rated]
ratings   = np.array([x[1] for x in rated])
Q_rated   = algo_full.qi[inner_ids]
b_rated   = algo_full.bi[inner_ids]
residuals = ratings - mu - b_rated

n_factors = Q_rated.shape[1]
A  = Q_rated.T @ Q_rated + LAMBDA * np.eye(n_factors)
bv = Q_rated.T @ residuals
pu = np.linalg.solve(A, bv)
print(f"Vecteur pu — norme : {np.linalg.norm(pu):.4f}")

rated_inner = set(inner_ids)
all_scores  = mu + algo_full.bi + algo_full.qi @ pu
candidates  = [
    (int(ts.to_raw_iid(i)), float(np.clip(all_scores[i], 0.5, 5.0)))
    for i in range(ts.n_items) if i not in rated_inner
]
candidates.sort(key=lambda x: -x[1])

print("\nTop 10 recommandations (folding-in) :")
for movie_id, score in candidates[:10]:
    title = df_items.loc[movie_id, C.LABEL_COL] if movie_id in df_items.index else "?"
    print(f"  movie_id={movie_id:6d}  score={score:.3f}  {title}")

# 8. Générer l'artefact backend

In [ ]:
# Entraîner le modèle final avec les meilleurs paramètres trouvés par GridSearchCV
svd_algo = SVD(**best_params, random_state=1)
svd_algo.fit(trainset)

os.makedirs("backend/artifacts", exist_ok=True)
artifact_path = "backend/artifacts/svd_model.pkl"
with open(artifact_path, "wb") as f:
    pickle.dump(svd_algo, f)

print(f"Artefact sauvegardé : {artifact_path}")
print(f"  Paramètres : {best_params}")
print(f"  n_items    : {trainset.n_items}")
print(f"  n_users    : {trainset.n_users}")

# 9. Tester le modèle backend en isolation

In [ ]:
from backend.models.svd import load, recommend

load()

test_ratings = {1: 4.0, 2: 3.5, 3: 5.0, 4: 3.0, 5: 4.5}
recs = recommend(test_ratings, n=10)

print(f"{len(recs)} recommandations reçues")
for movie_id, score in recs[:5]:
    title = df_items.loc[movie_id, C.LABEL_COL] if movie_id in df_items.index else "?"
    print(f"  movie_id={movie_id:6d}  score={score:.3f}  {title}")

assert len(recs) == 10
assert all(isinstance(mid, int) for mid, _ in recs)
assert all(0.5 <= s <= 5.0 for _, s in recs)
assert [s for _, s in recs] == sorted([s for _, s in recs], reverse=True)
print("\nToutes les vérifications passées.")